# Getting started
## An introduction to momepy

Momepy is a library for quantitative analysis of urban form - urban morphometrics. It is built on top of [GeoPandas](http://geopandas.org), [PySAL](http://pysal.org) and [networkX](http://networkx.github.io).

Some of the functionality that momepy offers:

- Measuring [dimensions](../api.rst#measuring-dimension) of morphological elements, their parts, and aggregated structures.
- Quantifying [shapes](../api.rst#measuring-shape) of geometries representing a wide range of morphological features.
- Capturing [spatial distribution](../api.rst#measuring-spatial-distribution) of elements of one kind as well as relationships between different kinds.
- Computing density and other types of [intensity](../api.rst#measuring-intensity) characters.
- Calculating [diversity](../api.rst#measuring-diversity) of various aspects of urban form.
- Capturing [connectivity](../api.rst#measuring-connectivity) of urban street networks
- Generating relational [elements](../api.rst#managing-morphological-elements) of urban form (e.g. morphological tessellation)

Momepy aims to provide a wide range of tools for a systematic and exhaustive analysis of urban form. It can work with a wide range of elements, while focused on building footprints and street networks.

## Installation

Momepy can be easily installed using conda from `conda-forge`. For detailed installation instructions, please refer to the installation documentation.

```
conda install momepy -c conda-forge
```

### Dependencies

To run all examples in this notebook, you will need `osmnx` to download data from OpenStreetMap. You can install all using the following commands:

```
conda install osmnx -c conda-forge
```

## Simple examples

Here are simple examples using embedded `bubenec` dataset (part of the Bubeneč neighborhood in Prague).

In [ ]:
import geopandas as gpd
import momepy

Morphometric analysis using momepy usually starts with GeoPandas GeoDataFrame containing morphological elements. In this case, we will begin with buildings represented by their footprints. 

We have imported `momepy`, `geopandas` to handle the spatial data, and `matplotlib` to get a bit more control over plotting. To load `bubenec` data, we need to get retrieve correct path using `momepy.datasets.get_path("bubenec")`. The dataset itself is a GeoPackage with more layers, and now we want `buildings`.

In [ ]:
buildings = gpd.read_file(
    momepy.datasets.get_path("bubenec"), layer="buildings"
)

In [ ]:
ax = buildings.plot(figsize=(8, 8))
ax.set_axis_off()

Momepy defines function for each measurable character. To illustrate how momepy works, we can try to measure a few simple characters.

### Equivalent rectangular index

`momepy.equivalent_rectangular_index` is an example of a morphometric character capturing the shape of each object. It can be calculated using a simple function consuimg GeoPandas object:

In [ ]:
blg_eri = momepy.equivalent_rectangular_index(buildings)

The result is a `pandas.Series`.

In [ ]:
blg_eri.head()

In a typical worflow, you add the result as a new column of the GeoDataFrame

In [ ]:
buildings["eri"] = momepy.equivalent_rectangular_index(buildings)

Check how it looks.

In [ ]:
ax = buildings.plot("eri", legend=True, figsize=(8, 8))
ax.set_axis_off()

## Perimeter wall length

Geometric elements are rarely considered individually. With urban fabric forming blocks like the one used here, you may be interseted in a length of the perceived perimeter wall of such a block. Note that all buildings forming the same wall share the resulting value.

In [ ]:
buildings["wall_length"] = momepy.perimeter_wall(buildings)
buildings.head()

In [ ]:
ax = buildings.plot("wall_length", legend=True, figsize=(8, 8))
ax.set_axis_off()

## Capturing relation between different elements

Urban form is a complex entity that needs to be represented by multiple morphological elements, and we need to be able to describe the relationship between them. With the absence of plots for our `bubenec` case, we can use [morphological tessellation](elements/tessellation.ipynb) as the smallest spatial division.

In [ ]:
tessellation = momepy.morphological_tessellation(
    buildings, clip=momepy.buffered_limit(buildings, buffer=50)
)
tessellation.head()

In [ ]:
ax = tessellation.plot(edgecolor="white", figsize=(8, 8))
buildings.plot(ax=ax, color="white", alpha=0.5)
ax.set_axis_off()

### Coverage area ratio

Now we can calculate how big part of each tessellation cell is covered by a related building. This is straightforward without a need for an additional function, because the two have matching index. Buildings have the same `index` as related tessellation cells, which is used to link both together (see [Data Structure](data_structure.rst)).

In [ ]:
tessellation["CAR"] = buildings.area / tessellation.area

In [ ]:
ax = tessellation.plot(
    "CAR",
    edgecolor="white",
    legend=True,
    scheme="NaturalBreaks",
    k=10,
    legend_kwds={"loc": "lower left"},
    figsize=(8, 8),
)
buildings.plot(ax=ax, color="white", alpha=0.5)
ax.set_axis_off()

Finally, to cover the last of the essential elements, we import the street network. 

In [ ]:
streets = gpd.read_file(momepy.datasets.get_path("bubenec"), layer="streets")

In [ ]:
ax = tessellation.plot(edgecolor="white", linewidth=0.2, figsize=(8, 8))
buildings.plot(ax=ax, color="white", alpha=0.5)
streets.plot(ax=ax, color="black")
ax.set_axis_off()

### Street profile

`momepy.street_profile` captures the relations between the segments of the street network and buildings. 

In [ ]:
profile = momepy.street_profile(streets, buildings)
profile.head()

We have now captured multiple characters of the street profile. As we did not specify building height (we do not know it for our data), we have widths (mean) - `profile["width"]`, standard deviation of widths (along the segment) - `profile["width_deviation"]`,  and the degree of openness - `profile["openness"]`.

In [ ]:
streets["width"] = profile["width"]
streets["openness"] = profile["openness"]

In [ ]:
ax = buildings.plot()
streets.plot("width", ax=ax, legend=True, figsize=(8, 8))
ax.set_axis_off()
_ = ax.set_title("width")

In [ ]:
ax = buildings.plot()
streets.plot("openness", ax=ax, legend=True, figsize=(8, 8))
ax.set_axis_off()
_ = ax.set_title("openness")

## Using OpenStreetMap data

In some cases (based on the completeness of OSM), we can use OpenStreetMap data without the need to save them to file and read them via GeoPandas. We can use OSMnx to retrieve them directly. In this example, we will download the building footprint of Kahla in Germany and project it to projected CRS. Momepy expects that all GeoDataFrames have the same (projected) CRS.

In [ ]:
import osmnx as ox

gdf = ox.features_from_place("Kahla, Germany", tags={"building": True})
gdf_projected = ox.projection.project_gdf(gdf)

In [ ]:
ax = gdf_projected.plot(figsize=(8, 8))
ax.set_axis_off()

Now we are in the same situation as we were above, and we can start our morphometric analysis as illustrated on the equivalent rectangular index below.

In [ ]:
gdf_projected["eri"] = momepy.equivalent_rectangular_index(gdf_projected)

In [ ]:
ax = gdf_projected.plot(
    "eri", legend=True, scheme="NaturalBreaks", k=10, figsize=(8, 8)
)
ax.set_axis_off()

## Next steps

Now we have basic knowledge of what momepy is and how it works. It is time to [install](../install.rst) momepy (if you haven't done so yet) and browse the rest of this user guide to see more examples. Once done, head to the [API reference](../api.rst) to see the full extent of characters momepy can capture and find those you need for your research.